# Myllia: Bilinear + h5ad + STRING v12 (seq + net + protein.links)

This notebook merges:
- your optimized bilinear baseline (delta-SVD + h5ad rescue for missing perts)
- STRING v12 sequence + network embeddings (aliases -> protein -> gene mean)
- STRING v12 **protein.links** PPI embeddings (protein.links + protein.info -> gene graph -> SVD)

Default: use **protein.links config B** (the one that won the A/B test) as a **pert-side boost**, and keep the seq/net blocks as extra gated inputs.

In [1]:
import os, re, json
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

import anndata as ad
import scanpy as sc
from scipy import sparse as sp

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.preprocessing import normalize as sk_normalize

import torch
import torch.nn as nn

from myllia_metric import myllia_score

# =====================
# Repro + device
# =====================
SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

# =====================
# Embedding dims + model capacity
# =====================
EMB_DIM_PERT = 128  # pert gene embedding dim
EMB_DIM_OUT  = 128  # output gene embedding dim
RANK_R = 32

# =====================
# Optim / training
# =====================
DROPOUT = 0.10
LR = 2e-3 * 0.6
WD = 1e-4
BATCH_GENES = 16

EXPERIMENT_MODE = "screen"  # "screen" or "full"

N_SPLITS = 8
if EXPERIMENT_MODE == "screen":
    EPOCHS = 200
    PATIENCE = 8
else:
    EPOCHS = 400
    PATIENCE = 12

EVAL_EVERY = 5
GRAD_CLIP = 1.0

# alpha sweep (maximize)
ALPHA_GRID = np.linspace(0.0, 0.9, 46).astype(np.float32)  # step 0.02

# metric gate params (must match competition)
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

ROOT = Path(".")
H5AD_PATH = ROOT / "data" / "training_cells.h5ad"  # used ONLY for perts not in gene_columns

# =====================
# STRING protein.links config B (the one you picked)
# =====================
LINKS_CFG_B = dict(
    score_min=400,
    topk=200,
    edge_gamma=2.0,
    beta2=1.0,
    beta3=0.5,
)

# Cache
CACHE_DIR = ROOT / "cache" / "string_v12"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("device:", device, "EXPERIMENT_MODE:", EXPERIMENT_MODE)


device: cuda EXPERIMENT_MODE: screen


In [2]:
# -----------------------
# Scoring wrapper
# -----------------------
def score_delta(y_true, y_pred):
    # accepts either numpy arrays or pandas frames, same as official helper use
    return myllia_score(y_true, y_pred)



In [3]:
# -----------------------
# Load competition files (robust to the two common folder layouts)
# -----------------------
MEANS_PATH_CAND = [
    ROOT / "data" / "training_data_means.csv",
    ROOT / "Data" / "training_data_means.csv",
]
GT_PATH_CAND = [
    ROOT / "data" / "training_data_ground_truth_table.csv",
    ROOT / "Data" / "training_data_ground_truth_table.csv",
]
VALMAP_PATH_CAND = [
    ROOT / "data" / "pert_ids_val.csv",
    ROOT / "Data" / "pert_ids_val.csv",
    ROOT / "data" / "validation_mapping.csv",
    ROOT / "Data" / "validation_mapping.csv",
]
SUB_PATH_CAND = [
    ROOT / "data" / "sample_submission.csv",
    ROOT / "Data" / "sample_submission.csv",
]

def pick_existing(cands):
    for p in cands:
        if p.exists():
            return p
    # last resort: search by name
    for p in ROOT.rglob(Path(cands[0]).name):
        return p
    return None

TRAIN_MEANS_PATH = pick_existing(MEANS_PATH_CAND)
GT_PATH = pick_existing(GT_PATH_CAND)
VALMAP_PATH = pick_existing(VALMAP_PATH_CAND)
SUB_PATH = pick_existing(SUB_PATH_CAND)

if TRAIN_MEANS_PATH is None:
    raise FileNotFoundError("Could not find training_data_means.csv")
if GT_PATH is None:
    raise FileNotFoundError("Could not find training_data_ground_truth_table.csv")
if VALMAP_PATH is None:
    raise FileNotFoundError("Could not find pert_ids_val.csv / validation_mapping.csv")
if SUB_PATH is None:
    raise FileNotFoundError("Could not find sample_submission.csv")

df_means = pd.read_csv(TRAIN_MEANS_PATH)
df_gt    = pd.read_csv(GT_PATH)
df_valmap= pd.read_csv(VALMAP_PATH)
df_sub   = pd.read_csv(SUB_PATH)

# Identify perturbation column in means
pert_col = None
for c in ["pert_symbol", "pert", "gene", "target", "sgrna_symbol", "perturbation"]:
    if c in df_means.columns:
        pert_col = c
        break
if pert_col is None:
    raise ValueError(f"Couldn't find perturbation column in {TRAIN_MEANS_PATH.name}. Columns: {df_means.columns.tolist()[:20]}")

gene_columns = [c for c in df_means.columns if c != pert_col]

# Baseline row (non-targeting) if present
pvals = df_means[pert_col].astype(str).str.lower()
baseline_mask = pvals.isin(["non-targeting", "non_targeting", "non targeting", "control", "ctrl", "nt"])

if baseline_mask.any():
    base_rows = df_means.loc[baseline_mask, gene_columns].to_numpy(np.float32)
    x_base = base_rows.mean(axis=0).astype(np.float32)
    df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
else:
    # fallback: baseline = mean of training means (not ideal, but keeps notebook runnable)
    x_base = df_means[gene_columns].to_numpy(np.float32).mean(axis=0).astype(np.float32)
    df_train = df_means.reset_index(drop=True)

train_genes = df_train[pert_col].astype(str).reset_index(drop=True)

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :]  # (N, G) delta vs non-targeting baseline

# baseline delta for shrink (vector)
delta_baseline = D_train.mean(axis=0).astype(np.float32)

# pert_id -> gene symbol mapping (for leaderboard ids)
val_map = {}
if "pert_id" in df_valmap.columns:
    if "pert" in df_valmap.columns:
        val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))
    elif "pert_symbol" in df_valmap.columns:
        val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert_symbol"].astype(str)))

val_targets = []
if "pert" in df_valmap.columns:
    val_targets = df_valmap["pert"].astype(str).tolist()
elif "pert_symbol" in df_valmap.columns:
    val_targets = df_valmap["pert_symbol"].astype(str).tolist()

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))
print("means path:", TRAIN_MEANS_PATH)
print("gt path:", GT_PATH)
print("valmap path:", VALMAP_PATH)


Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60
means path: data\training_data_means.csv
gt path: data\training_data_ground_truth_table.csv
valmap path: data\pert_ids_val.csv


## Baseline embeddings

We build your **delta-SVD** embeddings from `D_train`, then (only for perts missing from `gene_columns`) we create a proxy embedding using control-cell coexpression from `training_cells.h5ad`.

In [4]:
# -----------------------
# Build baseline gene embeddings (delta-SVD) + optional h5ad rescue for missing perts
# -----------------------
geneU = pd.Index([str(g).upper() for g in gene_columns])

# signed log transform of deltas for SVD stability
D_log = np.sign(D_train) * np.log1p(np.abs(D_train))

# SVD on (N,G); use components_ as gene embeddings
svd_k = max(EMB_DIM_PERT, EMB_DIM_OUT)
svd = TruncatedSVD(n_components=svd_k, random_state=SEED)
svd.fit(D_log)  # components_ : (k, G)
gene_emb_all = svd.components_.T.astype(np.float32)  # (G, k)

# l2 normalize rows
gene_emb_all = gene_emb_all / (np.linalg.norm(gene_emb_all, axis=1, keepdims=True) + 1e-12)

gene2emb_pert = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_PERT].copy()
                 for i in range(len(gene_columns))}
gene2emb_out  = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_OUT ].copy()
                 for i in range(len(gene_columns))}

emb_fallback_pert = gene_emb_all[:, :EMB_DIM_PERT].mean(axis=0).astype(np.float32)
emb_fallback_out  = gene_emb_all[:, :EMB_DIM_OUT ].mean(axis=0).astype(np.float32)

missing_emb_pert = {}

def emb_pert(g: str) -> np.ndarray:
    gU = str(g).upper()
    if gU in gene2emb_pert:
        return gene2emb_pert[gU]
    if gU in missing_emb_pert:
        return missing_emb_pert[gU]
    return emb_fallback_pert

def emb_out(g: str) -> np.ndarray:
    return gene2emb_out.get(str(g).upper(), emb_fallback_out)

# Output gene embeddings in the exact output gene order
U_out = np.vstack([emb_out(g) for g in gene_columns]).astype(np.float32)    # (G, d_out)

# Pert embeddings for your training perts (fallback if not in gene_columns)
Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)  # (N, d_pert)

print("U_out:", U_out.shape, "Z_train:", Z_train.shape)

# Coverage visibility
missing_train = [g for g in train_genes.tolist() if str(g).upper() not in geneU]
missing_val = [g for g in val_targets if str(g).upper() not in geneU]
if missing_train:
    print(f"train perts not in gene_columns: {len(missing_train)}. Example: {missing_train[:12]}")
if missing_val:
    print(f"val perts not in gene_columns: {len(missing_val)}. Example: {missing_val[:12]}") 

# h5ad rescue embeddings for missing perts: embed by control-cell coexpression
missing_all = sorted(set([str(x).upper() for x in (missing_train + missing_val)]))

if len(missing_all) > 0 and H5AD_PATH.exists():
    try:
        print("[h5ad] building embeddings for missing perts:", len(missing_all))
        adata = ad.read_h5ad(str(H5AD_PATH))

        # pick perturbation column
        pert_col = None
        for c in ["sgrna_symbol", "pert_symbol", "pert", "perturbation", "gene", "target_gene"]:
            if c in adata.obs.columns:
                pert_col = c
                break
        if pert_col is None:
            raise ValueError("Could not find perturbation column in h5ad obs.")

        Xc = adata.X
        if not sp.issparse(Xc):
            Xc = sp.csr_matrix(Xc)
        else:
            Xc = Xc.tocsr()

        # normalize ALL genes: CPM10K then log2(1+x)
        cell_sum = np.asarray(Xc.sum(axis=1)).ravel().astype(np.float64)
        scale = (10000.0 / np.clip(cell_sum, 1.0, None)).astype(np.float64)

        Xn = Xc.multiply(scale[:, None]).tocsr()
        Xn.data = np.log1p(Xn.data) / np.log(2.0)

        ctrl_mask = (adata.obs[pert_col].astype(str).to_numpy() == "non-targeting")
        if int(ctrl_mask.sum()) == 0:
            raise ValueError("No non-targeting control cells found in h5ad.")

        var = {str(g).upper(): i for i, g in enumerate(adata.var_names.astype(str).to_numpy())}

        # indices for the output genes in the full gene space
        out_idx = np.array([var[str(g).upper()] for g in gene_columns if str(g).upper() in var], dtype=np.int64)
        if len(out_idx) != len(gene_columns):
            miss_out = [g for g in gene_columns if str(g).upper() not in var]
            raise ValueError(f"{len(miss_out)} output genes missing from h5ad var_names. Example: {miss_out[:10]}") 

        Xout = Xn[ctrl_mask][:, out_idx]  # (n_ctrl, G)
        if sp.issparse(Xout):
            Xout = Xout.toarray()
        Xout = Xout.astype(np.float32)

        mu = Xout.mean(axis=0, keepdims=True)
        sd = Xout.std(axis=0, keepdims=True) + 1e-6
        Xout_z = (Xout - mu) / sd

        # use the existing pert embedding space (from SVD on D_train): (G, d_pert)
        P_out = gene_emb_all[:, :EMB_DIM_PERT].astype(np.float32)

        topk = 256
        made = 0
        for gU in missing_all:
            if gU not in var:
                continue

            xg = Xn[ctrl_mask, var[gU]]
            if sp.issparse(xg):
                xg = xg.toarray().ravel().astype(np.float32)
            else:
                xg = np.asarray(xg).ravel().astype(np.float32)

            xg_z = (xg - float(xg.mean())) / (float(xg.std()) + 1e-6)

            # correlation with each output gene across control cells
            corr = (xg_z[:, None] * Xout_z).mean(axis=0)  # (G,)

            # take top-|corr| genes
            sel = np.argpartition(np.abs(corr), -topk)[-topk:]
            w = corr[sel].astype(np.float32)
            w = w / (np.linalg.norm(w) + 1e-6)

            # weighted combo in embedding space
            v = (w[:, None] * P_out[sel]).sum(axis=0).astype(np.float32)
            v = v / (np.linalg.norm(v) + 1e-12)

            missing_emb_pert[gU] = v
            made += 1

        print(f"[h5ad] made {made}/{len(missing_all)} missing pert embeddings")

        # rebuild Z_train after filling missing_emb_pert
        Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)

    except Exception as e:
        print("[h5ad] WARN: could not build missing embeddings:", repr(e))

# keep baseline copies
U_out_base = U_out.copy()
Z_train_base = Z_train.copy()


U_out: (5127, 80) Z_train: (80, 80)
train perts not in gene_columns: 8. Example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']
[h5ad] building embeddings for missing perts: 16
[h5ad] made 16/16 missing pert embeddings


## Loss (row-weighted weighted-L1-like)

Uses your gate structure and upweights rows with small baseline_wmae (those are punished harder by the metric).

In [5]:
def gate_smoothstep(x, a=GATE_A, b=GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def per_row_weighted_l1_like(delta_true: torch.Tensor, delta_pred: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)  # (N,G)
    err = torch.abs(delta_pred - delta_true)                        # (N,G)
    num = torch.sum(w * err, dim=1)                                 # (N,)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)                 # (N,)
    return num / den                                                # (N,)

def weighted_l1_like_rowweighted(
    delta_true: torch.Tensor,     # (N,G)
    delta_pred: torch.Tensor,     # (N,G)
    baseline_wmae: torch.Tensor,  # (N,)
    *,
    eps: float = 1e-8,
    mode: str = "inv_sqrt",       # "inv", "inv_sqrt", "inv_log"
    clamp_min: float = 0.5,
    clamp_max: float = 3.0,
) -> torch.Tensor:
    per_row = per_row_weighted_l1_like(delta_true, delta_pred, eps=eps)  # (N,)

    b = baseline_wmae.to(delta_true.device).to(delta_true.dtype)
    if mode == "inv":
        w = 1.0 / (b + eps)
    elif mode == "inv_sqrt":
        w = 1.0 / torch.sqrt(b + eps)
    elif mode == "inv_log":
        w = 1.0 / torch.log1p(b + eps)
    else:
        raise ValueError(f"Unknown mode={mode}")

    w = torch.clamp(w, min=clamp_min, max=clamp_max)
    return torch.sum(w * per_row) / torch.clamp(torch.sum(w), min=eps)


## Tensors + baseline_wmae alignment

`baseline_wmae` is aligned to the **training rows**.

In [6]:
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

# -----------------------
# Align baseline_wmae to training rows (robust)
# -----------------------
def align_baseline_wmae(df_train: pd.DataFrame, df_gt: pd.DataFrame, train_pert_col: str):
    if "baseline_wmae" not in df_gt.columns:
        return np.ones((len(df_train),), dtype=np.float32)

    # 1) direct pert_id join
    if ("pert_id" in df_train.columns) and ("pert_id" in df_gt.columns):
        m = df_gt.groupby("pert_id")["baseline_wmae"].mean()
        return np.array([m.loc[str(pid)] for pid in df_train["pert_id"].astype(str).tolist()], dtype=np.float32)

    # 2) join on gene symbol / pert symbol if available
    for c in [train_pert_col, "pert", "pert_symbol", "gene", "target"]:
        if c in df_gt.columns:
            m = df_gt.groupby(df_gt[c].astype(str).str.upper())["baseline_wmae"].mean()
            return np.array([m.loc[str(p).upper()] for p in df_train[train_pert_col].astype(str).tolist()], dtype=np.float32)

    # 3) fallback: assume order aligns and length matches
    arr = df_gt["baseline_wmae"].to_numpy(dtype=np.float32)
    if len(arr) == len(df_train):
        return arr
    # fallback: mean
    return np.full((len(df_train),), float(np.mean(arr)), dtype=np.float32)

baseline_wmae = align_baseline_wmae(df_train, df_gt, pert_col)
baseline_wmae_t = torch.tensor(baseline_wmae, device=device, dtype=torch.float32)

# Torch tensors (base embeddings can be swapped via set_embeddings)
Uo_t = torch.tensor(U_out, device=device, dtype=torch.float32)     # (G, d_out)
Zt  = torch.tensor(Z_train, device=device, dtype=torch.float32)    # (N, d_pert)
Yt  = torch.tensor(Y, device=device, dtype=torch.float32)          # (N, G)

print("N:", N, "G:", G, "Uo_t:", tuple(Uo_t.shape), "Zt:", tuple(Zt.shape))


N: 80 G: 5127 Uo_t: (5127, 80) Zt: (80, 80)


In [11]:
# -----------------------
# CV helpers
# -----------------------
def apply_shrink(pred: np.ndarray, baseline_vec: np.ndarray, alpha: float) -> np.ndarray:
    a = float(alpha)
    return a * pred + (1.0 - a) * baseline_vec[None, :]

def set_embeddings(U_out_np: np.ndarray, Z_train_np: np.ndarray):
    global Uo_t, Zt
    assert U_out_np.shape[0] == G
    assert Z_train_np.shape[0] == N
    Uo_t = torch.tensor(U_out_np, device=device, dtype=torch.float32)
    Zt   = torch.tensor(Z_train_np, device=device, dtype=torch.float32)

def l2norm_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return (X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)).astype(np.float32)

# -----------------------
# Baseline model
# -----------------------
class BilinearDeltaModel(nn.Module):
    def __init__(self, d_pert, d_out, rank_r, dropout):
        super().__init__()
        self.proj_p = nn.Sequential(
            nn.Linear(d_pert, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.proj_o = nn.Sequential(
            nn.Linear(d_out, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = nn.Parameter(torch.zeros(1, device=dev))

    def forward(self, z_pert, u_out):
        p = self.proj_p(z_pert)    # (B,R)
        o = self.proj_o(u_out)     # (G,R)
        y = p @ o.T                # (B,G)
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

def train_one_fold(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_state = None
    best_va_pred = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            dt_b = Yt.index_select(0, b_t)
            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo_t).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]

            sc_best = -1e18
            a_best = 0.0
            for a in ALPHA_GRID:
                pred_a = apply_shrink(va_pred, delta_baseline, float(a))
                sc = score_delta(va_true, pred_a).score
                if sc > sc_best:
                    sc_best = sc
                    a_best = float(a)

            if sc_best > best_score:
                best_score = float(sc_best)
                best_alpha = float(a_best)
                best_epoch = int(epoch)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_va_pred = va_pred.copy()
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_state, best_va_pred

def run_cv_once(seed=SEED, tag="run"):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    oof_pred = np.zeros_like(Y, dtype=np.float32)
    oof_hit = np.zeros((N,), dtype=np.int32)

    fold_scores = []
    fold_alphas = []
    fold_epochs = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
        best_score, best_alpha, best_epoch, best_state, best_va_pred = train_one_fold(tr_idx, va_idx, seed=seed)

        fold_scores.append(float(best_score))
        fold_alphas.append(float(best_alpha))
        fold_epochs.append(int(best_epoch))

        oof_pred[va_idx] = best_va_pred
        oof_hit[va_idx] += 1

        print(f"[{tag}] fold {fold}: best_score={best_score:.6f} best_alpha={best_alpha:.3f} best_epoch={best_epoch}")

    if not np.all(oof_hit == 1):
        print(f"[{tag}] [warn] OOF coverage not 1 everywhere. min/max:", int(oof_hit.min()), int(oof_hit.max()))

    cv_mean = float(np.mean(fold_scores))
    cv_std  = float(np.std(fold_scores))
    med_ep  = int(np.median(fold_epochs))

    # global alpha on OOF
    best_global_alpha = 0.0
    best_global_score = -1e18
    for a in ALPHA_GRID:
        pred_a = apply_shrink(oof_pred, delta_baseline, float(a))
        sc = score_delta(Y, pred_a).score
        if sc > best_global_score:
            best_global_score = float(sc)
            best_global_alpha = float(a)

    out = {
        "name": tag,
        "mode": EXPERIMENT_MODE,
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "median_best_epoch": med_ep,
        "oof_alpha": best_global_alpha,
        "oof_score": float(best_global_score),
    }
    print(f"[{tag}] cv_mean={cv_mean:.6f} cv_std={cv_std:.6f} median_best_epoch={med_ep} oof_alpha={best_global_alpha:.3f} oof_score={best_global_score:.6f}")
    return out


## Baseline sanity check (delta-SVD + h5ad)

This should reproduce your baseline CV around ~0.145ish, depending on mode.

In [12]:
# Baseline run
set_embeddings(U_out_base, Z_train_base)
baseline_res = run_cv_once(seed=SEED, tag="baseline_current_embeddings")
baseline_res


[baseline_current_embeddings] fold 1: best_score=0.136483 best_alpha=0.740 best_epoch=20
[baseline_current_embeddings] fold 2: best_score=0.098667 best_alpha=0.700 best_epoch=20
[baseline_current_embeddings] fold 3: best_score=0.097575 best_alpha=0.580 best_epoch=20
[baseline_current_embeddings] fold 4: best_score=0.102033 best_alpha=0.720 best_epoch=20
[baseline_current_embeddings] fold 5: best_score=0.144255 best_alpha=0.860 best_epoch=20
[baseline_current_embeddings] fold 6: best_score=0.176289 best_alpha=0.700 best_epoch=20
[baseline_current_embeddings] fold 7: best_score=0.160867 best_alpha=0.680 best_epoch=15
[baseline_current_embeddings] fold 8: best_score=0.118115 best_alpha=0.700 best_epoch=15
[baseline_current_embeddings] cv_mean=0.129285 cv_std=0.028047 median_best_epoch=20 oof_alpha=0.700 oof_score=0.127600


{'name': 'baseline_current_embeddings',
 'mode': 'screen',
 'cv_mean': 0.12928542067782822,
 'cv_std': 0.028047263312279917,
 'median_best_epoch': 20,
 'oof_alpha': 0.699999988079071,
 'oof_score': 0.12759985202479918}

## STRING v12 protein.links embeddings

We build a gene-level graph from `protein.links` using `protein.info` to map ENSP IDs to preferred gene symbols. Then we do a multihop diffusion mix (1-hop + beta2*2-hop + beta3*3-hop), SVD to low-dim, and normalize.

Important: for your winning setup, we **only fuse on the pert side** via concat+SVD using a mixer fitted on the training perts. Output side stays baseline.

In [13]:
# -----------------------
# STRING helpers
# -----------------------
def find_file(filename: str) -> Path:
    candidates = [
        ROOT / "data" / filename,
        ROOT / "Data" / filename,
        ROOT / "external" / "string" / filename,
        ROOT / "external" / filename,
        ROOT / "string" / filename,
        ROOT / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    for p in ROOT.rglob(filename):
        return p
    raise FileNotFoundError(f"Could not find {filename} under {ROOT.resolve()}")


In [14]:
# -----------------------
# STRING protein.links -> gene embeddings
# -----------------------
def load_ensp_to_symbol_from_protein_info(info_path: Path) -> dict:
    # expected columns: protein_external_id / string_protein_id, preferred_name, annotation ...
    df = pd.read_csv(info_path, sep="\t", comment="#")
    if df.shape[1] < 2:
        df = pd.read_csv(info_path, sep=r"\s+", comment="#")

    cols = list(df.columns)
    if cols and str(cols[0]).startswith("#"):
        df = df.rename(columns={cols[0]: str(cols[0]).lstrip("#")})
        cols = list(df.columns)

    id_col = cols[0]
    name_col = cols[1]

    prot = df[id_col].astype(str).to_numpy()
    name = df[name_col].astype(str).to_numpy()

    # strip "9606." prefix safely
    prot = np.char.replace(prot.astype("U"), "9606.", "")
    name = np.char.upper(name.astype("U"))

    return dict(zip(prot.tolist(), name.tolist()))

def csr_topk(A_csr: sp.csr_matrix, k: int) -> sp.csr_matrix:
    A = A_csr.tocsr(copy=True)
    A.sort_indices()
    indptr, indices, data = A.indptr, A.indices, A.data

    rr, cc, dd = [], [], []
    for i in range(A.shape[0]):
        s, e = indptr[i], indptr[i+1]
        if e <= s:
            continue
        row_idx = indices[s:e]
        row_dat = data[s:e]
        if (e - s) > k:
            sel = np.argpartition(row_dat, -k)[-k:]
            row_idx = row_idx[sel]
            row_dat = row_dat[sel]
        rr.append(np.full(len(row_idx), i, dtype=np.int32))
        cc.append(row_idx.astype(np.int32))
        dd.append(row_dat.astype(np.float32))

    if not rr:
        return sp.csr_matrix(A.shape, dtype=np.float32)

    r = np.concatenate(rr); c = np.concatenate(cc); d = np.concatenate(dd)
    return sp.coo_matrix((d, (r, c)), shape=A.shape).tocsr()

def build_adj_from_protein_links(
    links_path: Path,
    ensp2gi: dict,
    n_genes: int,
    score_min: int = 700,
    topk: int = 200,
    edge_gamma: float = 1.0,
    chunksize: int = 2_000_000,
    cache_npz=None,
) -> sp.csr_matrix:
    if cache_npz is not None and cache_npz.exists():
        print("[cache] load adj:", cache_npz)
        return sp.load_npz(cache_npz).tocsr()

    usecols = ["protein1", "protein2", "combined_score"]
    dtypes = {"protein1":"string", "protein2":"string", "combined_score":np.int32}

    rows_parts, cols_parts, dat_parts = [], [], []
    it = pd.read_csv(links_path, sep=r"\s+", usecols=usecols, dtype=dtypes, chunksize=chunksize)

    for ci, chunk in enumerate(it, 1):
        s = chunk["combined_score"].to_numpy()
        m_score = s >= int(score_min)
        if not m_score.any():
            continue

        p1 = chunk["protein1"].to_numpy()[m_score]
        p2 = chunk["protein2"].to_numpy()[m_score]

        # base weight in [0,1]
        w = (s[m_score].astype(np.float32) / 1000.0)
        if float(edge_gamma) != 1.0:
            w = np.power(w, float(edge_gamma)).astype(np.float32)

        # strip "9606." prefix
        p1 = np.char.replace(p1.astype("U"), "9606.", "")
        p2 = np.char.replace(p2.astype("U"), "9606.", "")

        # map to gene indices (vectorized via pandas)
        i = pd.Series(p1).map(ensp2gi).fillna(-1).to_numpy(np.int32)
        j = pd.Series(p2).map(ensp2gi).fillna(-1).to_numpy(np.int32)

        m = (i >= 0) & (j >= 0) & (i != j)
        if not m.any():
            continue

        rows_parts.append(i[m])
        cols_parts.append(j[m])
        dat_parts.append(w[m])

        if ci % 5 == 0:
            kept = sum(len(x) for x in rows_parts)
            print(f"[parse] chunks={ci} kept_edges={kept:,}")

    if not rows_parts:
        A = sp.csr_matrix((n_genes, n_genes), dtype=np.float32)
    else:
        r = np.concatenate(rows_parts)
        c = np.concatenate(cols_parts)
        d = np.concatenate(dat_parts)
        A = sp.coo_matrix((d, (r, c)), shape=(n_genes, n_genes), dtype=np.float32).tocsr()

    # sym + prune
    A = A + A.T
    A.sum_duplicates()

    if topk is not None and int(topk) > 0:
        A = csr_topk(A, int(topk))
        A = A + A.T
        A.sum_duplicates()

    # row normalize to make a transition-ish matrix
    A = sk_normalize(A, norm="l1", axis=1)

    if cache_npz is not None:
        print("[cache] save adj:", cache_npz)
        sp.save_npz(cache_npz, A)

    return A

def embed_from_adj(A_csr: sp.csr_matrix, emb_dim: int = 128, seed: int = SEED) -> np.ndarray:
    svd = TruncatedSVD(n_components=int(emb_dim), random_state=int(seed))
    U = svd.fit_transform(A_csr).astype(np.float32)
    return l2norm_rows(U)

def embed_from_adj_multihop(
    A_csr: sp.csr_matrix,
    emb_dim: int = 128,
    beta2: float = 1.0,
    beta3: float = 0.0,
    seed: int = SEED,
    a2_topk: int = 400,
    a3_topk: int = 400,
) -> np.ndarray:
    A1 = sk_normalize(A_csr, norm="l1", axis=1)

    A2 = (A1 @ A1).tocsr()
    A2.sum_duplicates()
    if a2_topk is not None and int(a2_topk) > 0:
        A2 = csr_topk(A2, int(a2_topk))
        A2 = sk_normalize(A2, norm="l1", axis=1)

    if float(beta3) != 0.0:
        A3 = (A2 @ A1).tocsr()
        A3.sum_duplicates()
        if a3_topk is not None and int(a3_topk) > 0:
            A3 = csr_topk(A3, int(a3_topk))
            A3 = sk_normalize(A3, norm="l1", axis=1)
        Amix = (A1 + float(beta2) * A2 + float(beta3) * A3).tocsr()
    else:
        Amix = (A1 + float(beta2) * A2).tocsr()

    Amix.sum_duplicates()
    return embed_from_adj(Amix, emb_dim=emb_dim, seed=seed)

def build_string_links_universe_embeddings(
    score_min: int,
    topk: int,
    edge_gamma: float,
    beta2: float,
    beta3: float,
    emb_dim: int,
):
    links_path = find_file("9606.protein.links.v12.0.txt")
    info_path  = find_file("9606.protein.info.v12.0.txt")

    # gene universe: outputs + train perts + val perts
    universe = sorted(set([str(g).upper() for g in gene_columns] +
                          [str(g).upper() for g in train_genes.tolist()] +
                          [str(g).upper() for g in val_targets]))

    g2i = {g:i for i,g in enumerate(universe)}
    i_out  = np.array([g2i[str(g).upper()] for g in gene_columns], dtype=np.int32)
    i_pert = np.array([g2i.get(str(g).upper(), -1) for g in train_genes.tolist()], dtype=np.int32)

    ensp2sym = load_ensp_to_symbol_from_protein_info(info_path)

    # protein -> gene index (only if gene in universe)
    ensp2gi = {}
    for ensp, sym in ensp2sym.items():
        gi = g2i.get(sym, None)
        if gi is not None:
            ensp2gi[ensp] = gi

    cache_npz = CACHE_DIR / f"links_adj_smin{score_min}_topk{topk}_eg{edge_gamma:g}_n{len(universe)}.npz"
    A = build_adj_from_protein_links(
        links_path=links_path,
        ensp2gi=ensp2gi,
        n_genes=len(universe),
        score_min=int(score_min),
        topk=int(topk),
        edge_gamma=float(edge_gamma),
        chunksize=2_000_000,
        cache_npz=cache_npz,
    )

    cache_emb = CACHE_DIR / f"links_U_smin{score_min}_topk{topk}_eg{edge_gamma:g}_b2{beta2:g}_b3{beta3:g}_k{emb_dim}_n{len(universe)}.npy"
    if cache_emb.exists():
        print("[cache] load emb:", cache_emb)
        U = np.load(cache_emb).astype(np.float32)
    else:
        U = embed_from_adj_multihop(A, emb_dim=emb_dim, beta2=beta2, beta3=beta3, seed=SEED, a2_topk=topk, a3_topk=topk)
        print("[cache] save emb:", cache_emb)
        np.save(cache_emb, U)

    u_mean = U.mean(axis=0).astype(np.float32)

    U_out_str = U[i_out].astype(np.float32)  # (G, k)
    Z_train_str = np.vstack([(U[i] if i >= 0 else u_mean) for i in i_pert]).astype(np.float32)  # (N,k)

    pack = {
        "universe": universe,
        "g2i": g2i,
        "U": U,
        "u_mean": u_mean,
        "U_out": U_out_str,
        "Z_train": Z_train_str,
    }
    return pack


In [15]:
# -----------------------
# Pert-only fusion: concat + SVD mixer fitted on training perts
# -----------------------
def fit_svd_mixer_on_train(Z_base_train: np.ndarray, Z_extra_train: np.ndarray, out_dim: int, seed: int):
    X = np.concatenate([Z_base_train, Z_extra_train], axis=1).astype(np.float32)
    svd = TruncatedSVD(n_components=int(out_dim), random_state=int(seed))
    svd.fit(X)
    return svd

def apply_svd_mixer(svd, Z_base: np.ndarray, Z_extra: np.ndarray) -> np.ndarray:
    X = np.concatenate([Z_base, Z_extra], axis=1).astype(np.float32)
    Z = svd.transform(X).astype(np.float32)
    return l2norm_rows(Z)

def build_Z_links_for_perts(perts, pack) -> np.ndarray:
    U = pack["U"]
    g2i = pack["g2i"]
    u_mean = pack["u_mean"]
    out = np.zeros((len(perts), U.shape[1]), dtype=np.float32)
    for i, p in enumerate(perts):
        gi = g2i.get(str(p).upper(), -1)
        out[i] = U[gi] if gi >= 0 else u_mean
    return out

# Build B config links pack
links_pack_B = build_string_links_universe_embeddings(
    score_min=LINKS_CFG_B["score_min"],
    topk=LINKS_CFG_B["topk"],
    edge_gamma=LINKS_CFG_B["edge_gamma"],
    beta2=LINKS_CFG_B["beta2"],
    beta3=LINKS_CFG_B["beta3"],
    emb_dim=EMB_DIM_PERT,
)

Z_links_train_B = links_pack_B["Z_train"].astype(np.float32)

svd_mix_pert_B = fit_svd_mixer_on_train(Z_train_base, Z_links_train_B, out_dim=EMB_DIM_PERT, seed=SEED)
Z_train_base_links_B = apply_svd_mixer(svd_mix_pert_B, Z_train_base, Z_links_train_B)

print("Z_train_base:", Z_train_base.shape, "Z_links_train_B:", Z_links_train_B.shape, "Z_train_base_links_B:", Z_train_base_links_B.shape)


[parse] chunks=5 kept_edges=210,369
[cache] save adj: cache\string_v12\links_adj_smin400_topk200_eg2_n5143.npz
[cache] save emb: cache\string_v12\links_U_smin400_topk200_eg2_b21_b30.5_k128_n5143.npy
Z_train_base: (80, 80) Z_links_train_B: (80, 128) Z_train_base_links_B: (80, 80)


In [16]:
# CV sanity: baseline vs links-pert-only (B config)
results = []
set_embeddings(U_out_base, Z_train_base)
results.append(run_cv_once(seed=SEED, tag="baseline_current_embeddings"))

set_embeddings(U_out_base, Z_train_base_links_B)
results.append(run_cv_once(seed=SEED, tag="linksB_mix_pert_only"))

pd.DataFrame(results).sort_values("oof_score", ascending=False).reset_index(drop=True)


[baseline_current_embeddings] fold 1: best_score=0.136483 best_alpha=0.740 best_epoch=20
[baseline_current_embeddings] fold 2: best_score=0.098667 best_alpha=0.700 best_epoch=20
[baseline_current_embeddings] fold 3: best_score=0.097575 best_alpha=0.580 best_epoch=20
[baseline_current_embeddings] fold 4: best_score=0.102033 best_alpha=0.720 best_epoch=20
[baseline_current_embeddings] fold 5: best_score=0.144255 best_alpha=0.860 best_epoch=20
[baseline_current_embeddings] fold 6: best_score=0.176289 best_alpha=0.700 best_epoch=20
[baseline_current_embeddings] fold 7: best_score=0.160867 best_alpha=0.680 best_epoch=15
[baseline_current_embeddings] fold 8: best_score=0.118115 best_alpha=0.700 best_epoch=15
[baseline_current_embeddings] cv_mean=0.129285 cv_std=0.028047 median_best_epoch=20 oof_alpha=0.700 oof_score=0.127600
[linksB_mix_pert_only] fold 1: best_score=0.158904 best_alpha=0.900 best_epoch=20
[linksB_mix_pert_only] fold 2: best_score=0.107467 best_alpha=0.840 best_epoch=20
[link

,name,mode,cv_mean,cv_std,median_best_epoch,oof_alpha,oof_score
0,linksB_mix_pert_only,screen,0.139635,0.031532,20,0.78,0.137149
1,baseline_current_embeddings,screen,0.129285,0.028047,20,0.70,0.127600


## STRING v12 sequence + network embeddings

These are the *other* STRING embeddings you already built: protein aliases map gene symbol -> proteins, then you average protein embeddings per gene.

We keep them as additional blocks with masks (missing genes just get zero + mask=0).

In [17]:
import h5py

STRING_DIR = ROOT / "external" / "string"
ALIASES_PATH = STRING_DIR / "9606.protein.aliases.v12.0.txt"
SEQ_H5_PATH  = STRING_DIR / "9606.protein.sequence.embeddings.v12.0.h5"
NET_H5_PATH  = STRING_DIR / "9606.protein.network.embeddings.v12.0.h5"

for p in [ALIASES_PATH, SEQ_H5_PATH, NET_H5_PATH]:
    print(("OK " if p.exists() else "MISS ") + str(p))


OK external\string\9606.protein.aliases.v12.0.txt
OK external\string\9606.protein.sequence.embeddings.v12.0.h5
OK external\string\9606.protein.network.embeddings.v12.0.h5


In [18]:
# -----------------------
# Aliases: gene symbol -> ENSP proteins (streaming)
# -----------------------
def build_gene_to_proteins_map(aliases_path: Path, needed_genesU: set):
    gene2prot = {}
    usecols = ["#string_protein_id", "alias", "source"]
    # Some files use different header; tolerate it.
    # We'll read without usecols first chunk to detect, then stream with correct names.
    head = pd.read_csv(aliases_path, sep="\t", nrows=5, comment=None)
    cols = list(head.columns)
    # normalize leading '#'
    if cols and str(cols[0]).startswith("#"):
        cols0 = str(cols[0]).lstrip("#")
        head = head.rename(columns={cols[0]: cols0})
        cols = list(head.columns)

    # Try best guesses for columns
    prot_col = None
    alias_col = None
    src_col = None
    for c in cols:
        cL = c.lower()
        if "protein" in cL and "id" in cL:
            prot_col = c
        if cL in ["alias", "preferred_name", "name"]:
            alias_col = c
        if "source" in cL:
            src_col = c
    if prot_col is None:
        prot_col = cols[0]
    if alias_col is None:
        alias_col = cols[1]
    if src_col is None and len(cols) >= 3:
        src_col = cols[2]

    # sources to prioritize for gene symbols
    allow_sources = set(["BLAST_UniProt_GN", "BLAST_UniProt_ID", "Ensembl_HGNC", "Ensembl_HGNC_curated", "Ensembl_UniProt", "HGNC", "UniProt_ID", "UniProt_GN"])

    chunksize = 2_000_000
    it = pd.read_csv(aliases_path, sep="\t", chunksize=chunksize)
    for ci, chunk in enumerate(it, 1):
        if str(chunk.columns[0]).startswith("#"):
            chunk = chunk.rename(columns={chunk.columns[0]: str(chunk.columns[0]).lstrip("#")})
        prot = chunk[prot_col].astype(str).to_numpy()
        alias = chunk[alias_col].astype(str).to_numpy()
        src = chunk[src_col].astype(str).to_numpy() if src_col in chunk.columns else np.array([""] * len(chunk), dtype=object)

        # strip species prefix and uppercase gene aliases
        prot = np.char.replace(prot.astype("U"), "9606.", "")
        aliasU = np.char.upper(alias.astype("U"))

        # filter on needed genes and allowed sources
        m = np.isin(aliasU, list(needed_genesU))
        if m.any():
            prot_m = prot[m]
            alias_m = aliasU[m]
            src_m = src[m]

            for p_id, gU, sU in zip(prot_m.tolist(), alias_m.tolist(), src_m.tolist()):
                if sU and (sU not in allow_sources):
                    continue
                gene2prot.setdefault(gU, []).append(p_id)

        if ci % 5 == 0:
            print(f"[aliases] chunks={ci} mapped_genes={len(gene2prot)}")

    # de-dup proteins per gene
    for gU in list(gene2prot.keys()):
        gene2prot[gU] = sorted(set(gene2prot[gU]))

    return gene2prot

# Needed genes universe for seq/net blocks
needed_genesU = set([str(g).upper() for g in gene_columns] +
                    [str(g).upper() for g in train_genes.tolist()] +
                    [str(g).upper() for g in val_targets])

print("needed genes:", len(needed_genesU))

cache_gene2prot = CACHE_DIR / "gene2prot_aliases_v12.json"
if cache_gene2prot.exists():
    gene2prot = json.loads(cache_gene2prot.read_text())
else:
    gene2prot = build_gene_to_proteins_map(ALIASES_PATH, needed_genesU)
    cache_gene2prot.write_text(json.dumps(gene2prot))

print("gene2prot size:", len(gene2prot))


needed genes: 5143
gene2prot size: 5002


In [19]:
# -----------------------
# Load protein embeddings H5 (sequence + network)
# -----------------------
def load_h5_embeddings(h5_path: Path):
    with h5py.File(h5_path, "r") as f:
        keys = list(f.keys())
        # common layouts:
        # - f["embeddings"] (N,d) and f["proteins"] (N,)
        # - f["X"] and f["ids"]
        for Xk in ["embeddings", "X", "data"]:
            if Xk in keys:
                X = f[Xk][:]
                break
        else:
            raise KeyError(f"No embeddings dataset found in {h5_path} keys={keys[:10]}")
        for Ik in ["proteins", "ids", "protein_ids"]:
            if Ik in keys:
                ids = f[Ik][:]
                break
        else:
            raise KeyError(f"No protein id dataset found in {h5_path} keys={keys[:10]}")
    ids = np.array([x.decode() if isinstance(x, (bytes, np.bytes_)) else str(x) for x in ids], dtype=object)
    ids = np.char.replace(ids.astype("U"), "9606.", "")
    return ids, X.astype(np.float32)

seq_ids, seq_emb_all = load_h5_embeddings(SEQ_H5_PATH)
net_ids, net_emb_all = load_h5_embeddings(NET_H5_PATH)

seq_idx = {pid: i for i, pid in enumerate(seq_ids.tolist())}
net_idx = {pid: i for i, pid in enumerate(net_ids.tolist())}

seq_dim = int(seq_emb_all.shape[1])
net_dim = int(net_emb_all.shape[1])

print("seq:", seq_emb_all.shape, "net:", net_emb_all.shape)


seq: (19699, 1024) net: (19699, 512)


In [20]:
# -----------------------
# Gene mean embedding from mapped proteins
# -----------------------
def mean_emb_for_gene(gU: str, gene2prot: dict, idx: dict, emb_all: np.ndarray):
    prots = gene2prot.get(gU, None)
    if not prots:
        return None
    vecs = []
    for p in prots:
        j = idx.get(p, None)
        if j is not None:
            vecs.append(emb_all[j])
    if not vecs:
        return None
    v = np.mean(np.stack(vecs, axis=0), axis=0).astype(np.float32)
    v = v / (np.linalg.norm(v) + 1e-12)
    return v

# Output-side embeddings + masks
U_seq = np.zeros((len(gene_columns), seq_dim), dtype=np.float32)
U_net = np.zeros((len(gene_columns), net_dim), dtype=np.float32)
U_seq_mask = np.zeros((len(gene_columns), 1), dtype=np.float32)
U_net_mask = np.zeros((len(gene_columns), 1), dtype=np.float32)

for i, g in enumerate(gene_columns):
    gU = str(g).upper()
    vs = mean_emb_for_gene(gU, gene2prot, seq_idx, seq_emb_all)
    vn = mean_emb_for_gene(gU, gene2prot, net_idx, net_emb_all)
    if vs is not None:
        U_seq[i] = vs
        U_seq_mask[i, 0] = 1.0
    if vn is not None:
        U_net[i] = vn
        U_net_mask[i, 0] = 1.0

# Pert-side embeddings + masks (training perts in exact order)
Z_seq = np.zeros((len(train_genes), seq_dim), dtype=np.float32)
Z_net = np.zeros((len(train_genes), net_dim), dtype=np.float32)
Z_seq_mask = np.zeros((len(train_genes), 1), dtype=np.float32)
Z_net_mask = np.zeros((len(train_genes), 1), dtype=np.float32)

for i, g in enumerate(train_genes.tolist()):
    gU = str(g).upper()
    vs = mean_emb_for_gene(gU, gene2prot, seq_idx, seq_emb_all)
    vn = mean_emb_for_gene(gU, gene2prot, net_idx, net_emb_all)
    if vs is not None:
        Z_seq[i] = vs
        Z_seq_mask[i, 0] = 1.0
    if vn is not None:
        Z_net[i] = vn
        Z_net_mask[i, 0] = 1.0

print("U_seq coverage:", float(U_seq_mask.mean()), "U_net coverage:", float(U_net_mask.mean()))
print("Z_seq coverage:", float(Z_seq_mask.mean()), "Z_net coverage:", float(Z_net_mask.mean()))

# Torch tensors
U_seq_t = torch.tensor(U_seq, device=device, dtype=torch.float32)
U_net_t = torch.tensor(U_net, device=device, dtype=torch.float32)
U_seq_mask_t = torch.tensor(U_seq_mask, device=device, dtype=torch.float32)
U_net_mask_t = torch.tensor(U_net_mask, device=device, dtype=torch.float32)

Z_seq_t = torch.tensor(Z_seq, device=device, dtype=torch.float32)
Z_net_t = torch.tensor(Z_net, device=device, dtype=torch.float32)
Z_seq_mask_t = torch.tensor(Z_seq_mask, device=device, dtype=torch.float32)
Z_net_mask_t = torch.tensor(Z_net_mask, device=device, dtype=torch.float32)


U_seq coverage: 0.9724985361099243 U_net coverage: 0.9724985361099243
Z_seq coverage: 0.987500011920929 Z_net coverage: 0.987500011920929


## Hybrid model: base (possibly base+links) + seq + net

The gate learns how much to use each block per-row and per-gene.

In [21]:
class HybridStringBilinearDeltaModel(nn.Module):
    """Bilinear predictor with 3 blocks on each side:
    - base embeddings (delta-SVD, optionally pert-side fused with protein.links)
    - seq embeddings (STRING sequence H5)
    - net embeddings (STRING network H5)

    Masks allow missing genes/perts to contribute 0.
    """
    def __init__(self, d_base_p, d_base_o, d_seq, d_net, rank_r, dropout):
        super().__init__()
        self.rank_r = rank_r

        # project each block to R
        self.proj_base_p = nn.Sequential(nn.Linear(d_base_p, rank_r), nn.GELU(), nn.Dropout(dropout))
        self.proj_seq_p  = nn.Sequential(nn.Linear(d_seq,     rank_r), nn.GELU(), nn.Dropout(dropout))
        self.proj_net_p  = nn.Sequential(nn.Linear(d_net,     rank_r), nn.GELU(), nn.Dropout(dropout))

        self.proj_base_o = nn.Sequential(nn.Linear(d_base_o, rank_r), nn.GELU(), nn.Dropout(dropout))
        self.proj_seq_o  = nn.Sequential(nn.Linear(d_seq,    rank_r), nn.GELU(), nn.Dropout(dropout))
        self.proj_net_o  = nn.Sequential(nn.Linear(d_net,    rank_r), nn.GELU(), nn.Dropout(dropout))

        # gates: per-row for pert blocks, per-gene for output blocks
        self.gate_p = nn.Sequential(nn.Linear(d_base_p + d_seq + d_net + 2, 64), nn.GELU(), nn.Linear(64, 3))
        self.gate_o = nn.Sequential(nn.Linear(d_base_o + d_seq + d_net + 2, 64), nn.GELU(), nn.Linear(64, 3))

        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = nn.Parameter(torch.zeros(1, device=dev))

    def forward(
        self,
        z_base, z_seq, z_net,
        u_base, u_seq, u_net,
        z_seq_mask, z_net_mask,
        u_seq_mask, u_net_mask,
    ):
        # project
        p_base = self.proj_base_p(z_base)
        p_seq  = self.proj_seq_p(z_seq)
        p_net  = self.proj_net_p(z_net)

        o_base = self.proj_base_o(u_base)
        o_seq  = self.proj_seq_o(u_seq)
        o_net  = self.proj_net_o(u_net)

        # apply masks by zeroing missing blocks
        p_seq = p_seq * z_seq_mask
        p_net = p_net * z_net_mask
        o_seq = o_seq * u_seq_mask
        o_net = o_net * u_net_mask

        # gate inputs (concat + masks)
        gp_in = torch.cat([z_base, z_seq, z_net, z_seq_mask, z_net_mask], dim=1)
        go_in = torch.cat([u_base, u_seq, u_net, u_seq_mask, u_net_mask], dim=1)

        w_p = torch.softmax(self.gate_p(gp_in), dim=1)  # (B,3)
        w_o = torch.softmax(self.gate_o(go_in), dim=1)  # (G,3)

        # weighted sums in rank space
        p = (w_p[:, [0]] * p_base) + (w_p[:, [1]] * p_seq) + (w_p[:, [2]] * p_net)   # (B,R)
        o = (w_o[:, [0]] * o_base) + (w_o[:, [1]] * o_seq) + (w_o[:, [2]] * o_net)   # (G,R)

        y = p @ o.T
        y = y + self.bias_gene[None, :] + self.bias_global
        return y


In [22]:
# -----------------------
# Hybrid CV
# -----------------------
def train_one_fold_hybrid(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = HybridStringBilinearDeltaModel(
        d_base_p=Zt.shape[1],
        d_base_o=Uo_t.shape[1],
        d_seq=U_seq_t.shape[1],
        d_net=U_net_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT,
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_state = None
    best_va_pred = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(
                Zt.index_select(0, b_t),
                Z_seq_t.index_select(0, b_t),
                Z_net_t.index_select(0, b_t),
                Uo_t,
                U_seq_t,
                U_net_t,
                Z_seq_mask_t.index_select(0, b_t),
                Z_net_mask_t.index_select(0, b_t),
                U_seq_mask_t,
                U_net_mask_t,
            )

            dt_b = Yt.index_select(0, b_t)
            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(
                    Zt.index_select(0, va_idx_t),
                    Z_seq_t.index_select(0, va_idx_t),
                    Z_net_t.index_select(0, va_idx_t),
                    Uo_t,
                    U_seq_t,
                    U_net_t,
                    Z_seq_mask_t.index_select(0, va_idx_t),
                    Z_net_mask_t.index_select(0, va_idx_t),
                    U_seq_mask_t,
                    U_net_mask_t,
                ).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]

            sc_best = -1e18
            a_best = 0.0
            for a in ALPHA_GRID:
                pred_a = apply_shrink(va_pred, delta_baseline, float(a))
                sc = score_delta(va_true, pred_a).score
                if sc > sc_best:
                    sc_best = sc
                    a_best = float(a)

            if sc_best > best_score:
                best_score = float(sc_best)
                best_alpha = float(a_best)
                best_epoch = int(epoch)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_va_pred = va_pred.copy()
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_state, best_va_pred

def run_cv_once_hybrid(seed=SEED, tag="hybrid"):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    oof_pred = np.zeros_like(Y, dtype=np.float32)
    oof_hit = np.zeros((N,), dtype=np.int32)

    fold_scores = []
    fold_alphas = []
    fold_epochs = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
        best_score, best_alpha, best_epoch, best_state, best_va_pred = train_one_fold_hybrid(tr_idx, va_idx, seed=seed)

        fold_scores.append(float(best_score))
        fold_alphas.append(float(best_alpha))
        fold_epochs.append(int(best_epoch))

        oof_pred[va_idx] = best_va_pred
        oof_hit[va_idx] += 1

        print(f"[{tag}] fold {fold}: best_score={best_score:.6f} best_alpha={best_alpha:.3f} best_epoch={best_epoch}")

    cv_mean = float(np.mean(fold_scores))
    cv_std  = float(np.std(fold_scores))
    med_ep  = int(np.median(fold_epochs))

    # global alpha on OOF
    best_global_alpha = 0.0
    best_global_score = -1e18
    for a in ALPHA_GRID:
        pred_a = apply_shrink(oof_pred, delta_baseline, float(a))
        sc = score_delta(Y, pred_a).score
        if sc > best_global_score:
            best_global_score = float(sc)
            best_global_alpha = float(a)

    out = {
        "name": tag,
        "mode": EXPERIMENT_MODE,
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "median_best_epoch": med_ep,
        "oof_alpha": best_global_alpha,
        "oof_score": float(best_global_score),
    }
    print(f"[{tag}] cv_mean={cv_mean:.6f} cv_std={cv_std:.6f} median_best_epoch={med_ep} oof_alpha={best_global_alpha:.3f} oof_score={best_global_score:.6f}")
    return out


## Compare variants

Four runs you actually care about:
- baseline bilinear
- bilinear + linksB pert-only fusion
- hybrid (baseline base) + seq/net
- hybrid + linksB pert-only fusion + seq/net

In [23]:
results2 = []

# (1) baseline bilinear
set_embeddings(U_out_base, Z_train_base)
results2.append(run_cv_once(seed=SEED, tag="baseline_bilinear"))

# (2) linksB bilinear
set_embeddings(U_out_base, Z_train_base_links_B)
results2.append(run_cv_once(seed=SEED, tag="linksB_bilinear"))

# (3) hybrid with baseline base
set_embeddings(U_out_base, Z_train_base)
results2.append(run_cv_once_hybrid(seed=SEED, tag="hybrid_seqnet"))

# (4) hybrid with linksB base
set_embeddings(U_out_base, Z_train_base_links_B)
results2.append(run_cv_once_hybrid(seed=SEED, tag="hybrid_seqnet_linksB"))

pd.DataFrame(results2).sort_values("oof_score", ascending=False).reset_index(drop=True)


[baseline_bilinear] fold 1: best_score=0.136483 best_alpha=0.740 best_epoch=20
[baseline_bilinear] fold 2: best_score=0.098667 best_alpha=0.700 best_epoch=20
[baseline_bilinear] fold 3: best_score=0.097575 best_alpha=0.580 best_epoch=20
[baseline_bilinear] fold 4: best_score=0.102033 best_alpha=0.720 best_epoch=20
[baseline_bilinear] fold 5: best_score=0.144255 best_alpha=0.860 best_epoch=20
[baseline_bilinear] fold 6: best_score=0.176289 best_alpha=0.700 best_epoch=20
[baseline_bilinear] fold 7: best_score=0.160867 best_alpha=0.680 best_epoch=15
[baseline_bilinear] fold 8: best_score=0.118115 best_alpha=0.700 best_epoch=15
[baseline_bilinear] cv_mean=0.129285 cv_std=0.028047 median_best_epoch=20 oof_alpha=0.700 oof_score=0.127600
[linksB_bilinear] fold 1: best_score=0.158904 best_alpha=0.900 best_epoch=20
[linksB_bilinear] fold 2: best_score=0.107467 best_alpha=0.840 best_epoch=20
[linksB_bilinear] fold 3: best_score=0.100596 best_alpha=0.640 best_epoch=20
[linksB_bilinear] fold 4: be

,name,mode,cv_mean,cv_std,median_best_epoch,oof_alpha,oof_score
0,hybrid_seqnet_linksB,screen,0.146768,0.032289,17,0.76,0.143844
1,hybrid_seqnet,screen,0.142920,0.033586,15,0.78,0.139910
2,linksB_bilinear,screen,0.139635,0.031532,20,0.78,0.137149
3,baseline_bilinear,screen,0.129285,0.028047,20,0.70,0.127600


## Multi-seed A/B for your chosen links config

This reproduces the tiny-delta (but real) advantage between A and B, except now for:
- baseline vs linksB (bilinear)
- hybrid vs hybrid+linksB

In [24]:
def ab_test(seeds, run_fn, nameA, setupA, nameB, setupB):
    rows = []
    for s in seeds:
        setupA()
        A = run_fn(seed=int(s), tag=nameA + f"_s{s}")["oof_score"]
        setupB()
        B = run_fn(seed=int(s), tag=nameB + f"_s{s}")["oof_score"]
        rows.append((s, A, B))
    df = pd.DataFrame(rows, columns=["seed", nameA, nameB]).set_index("seed")
    df["baseline"] = baseline_res["oof_score"] if isinstance(baseline_res, dict) else np.nan
    df["B_minus_A"] = df[nameB] - df[nameA]
    return df

SEEDS_AB = [6, 67, 6767]

# bilinear A=baseline, B=linksB
df_ab_bilinear = ab_test(
    SEEDS_AB,
    run_cv_once,
    nameA="base_bilinear",
    setupA=lambda: set_embeddings(U_out_base, Z_train_base),
    nameB="linksB_bilinear",
    setupB=lambda: set_embeddings(U_out_base, Z_train_base_links_B),
)

# hybrid A=hybrid, B=hybrid+linksB
df_ab_hybrid = ab_test(
    SEEDS_AB,
    run_cv_once_hybrid,
    nameA="hybrid",
    setupA=lambda: set_embeddings(U_out_base, Z_train_base),
    nameB="hybrid_linksB",
    setupB=lambda: set_embeddings(U_out_base, Z_train_base_links_B),
)

display(df_ab_bilinear)
display(df_ab_hybrid)

print("\nSummary bilinear:")
print(df_ab_bilinear[["base_bilinear", "linksB_bilinear", "B_minus_A"]].mean())

print("\nSummary hybrid:")
print(df_ab_hybrid[["hybrid", "hybrid_linksB", "B_minus_A"]].mean())


[base_bilinear_s6] fold 1: best_score=0.136483 best_alpha=0.740 best_epoch=20
[base_bilinear_s6] fold 2: best_score=0.098667 best_alpha=0.700 best_epoch=20
[base_bilinear_s6] fold 3: best_score=0.097575 best_alpha=0.580 best_epoch=20
[base_bilinear_s6] fold 4: best_score=0.102033 best_alpha=0.720 best_epoch=20
[base_bilinear_s6] fold 5: best_score=0.144255 best_alpha=0.860 best_epoch=20
[base_bilinear_s6] fold 6: best_score=0.176289 best_alpha=0.700 best_epoch=20
[base_bilinear_s6] fold 7: best_score=0.160867 best_alpha=0.680 best_epoch=15
[base_bilinear_s6] fold 8: best_score=0.118115 best_alpha=0.700 best_epoch=15
[base_bilinear_s6] cv_mean=0.129285 cv_std=0.028047 median_best_epoch=20 oof_alpha=0.700 oof_score=0.127600
[linksB_bilinear_s6] fold 1: best_score=0.158904 best_alpha=0.900 best_epoch=20
[linksB_bilinear_s6] fold 2: best_score=0.107467 best_alpha=0.840 best_epoch=20
[linksB_bilinear_s6] fold 3: best_score=0.100596 best_alpha=0.640 best_epoch=20
[linksB_bilinear_s6] fold 4:

,base_bilinear,linksB_bilinear,baseline,B_minus_A
seed,,,,
6,0.127600,0.137149,0.1276,0.009549
67,0.126338,0.133815,0.1276,0.007477
6767,0.127607,0.139190,0.1276,0.011583


,hybrid,hybrid_linksB,baseline,B_minus_A
seed,,,,
6,0.139910,0.143844,0.1276,0.003934
67,0.132223,0.133438,0.1276,0.001216
6767,0.141361,0.142868,0.1276,0.001507



Summary bilinear:
base_bilinear      0.127182
linksB_bilinear    0.136718
B_minus_A          0.009536
dtype: float64

Summary hybrid:
hybrid           0.137831
hybrid_linksB    0.140050
B_minus_A        0.002219
dtype: float64


## Submission (hybrid + linksB)

This refits on all 80 training perts with fixed epochs = median best epoch from CV, then predicts the sample submission rows.

Note: writing the CSV is commented by default.

In [ ]:
# -----------------------
# Build submission perts list (map pert_id -> gene symbol when possible)
# -----------------------
id_col = "pert_id" if "pert_id" in df_sub.columns else df_sub.columns[0]
sub_ids = df_sub[id_col].astype(str).tolist()

def pert_symbol_from_id(pid: str) -> str:
    return val_map.get(str(pid), str(pid))

sub_perts = [pert_symbol_from_id(pid) for pid in sub_ids]

# Base Z for perts: baseline vs linksB
def build_Z_base_baseline(perts):
    return np.vstack([emb_pert(p) for p in perts]).astype(np.float32)

def build_Z_base_linksB(perts):
    Zb = build_Z_base_baseline(perts)
    Zl = build_Z_links_for_perts(perts, links_pack_B)
    return apply_svd_mixer(svd_mix_pert_B, Zb, Zl)

# Seq/net Z + masks for perts
def build_Z_seqnet_and_masks(perts):
    Zs = np.zeros((len(perts), seq_dim), dtype=np.float32)
    Zn = np.zeros((len(perts), net_dim), dtype=np.float32)
    Ms = np.zeros((len(perts), 1), dtype=np.float32)
    Mn = np.zeros((len(perts), 1), dtype=np.float32)
    for i, p in enumerate(perts):
        gU = str(p).upper()
        vs = mean_emb_for_gene(gU, gene2prot, seq_idx, seq_emb_all)
        vn = mean_emb_for_gene(gU, gene2prot, net_idx, net_emb_all)
        if vs is not None:
            Zs[i] = vs
            Ms[i, 0] = 1.0
        if vn is not None:
            Zn[i] = vn
            Mn[i, 0] = 1.0
    return Zs, Zn, Ms, Mn

Z_base_sub = build_Z_base_linksB(sub_perts)  # default: linksB
Z_seq_sub, Z_net_sub, Z_seq_sub_mask, Z_net_sub_mask = build_Z_seqnet_and_masks(sub_perts)

Z_base_sub_t = torch.tensor(Z_base_sub, device=device, dtype=torch.float32)
Z_seq_sub_t  = torch.tensor(Z_seq_sub,  device=device, dtype=torch.float32)
Z_net_sub_t  = torch.tensor(Z_net_sub,  device=device, dtype=torch.float32)
Z_seq_sub_mask_t = torch.tensor(Z_seq_sub_mask, device=device, dtype=torch.float32)
Z_net_sub_mask_t = torch.tensor(Z_net_sub_mask, device=device, dtype=torch.float32)

print("Z_base_sub:", Z_base_sub.shape, "Z_seq_sub:", Z_seq_sub.shape, "Z_net_sub:", Z_net_sub.shape)


In [ ]:
# -----------------------
# Refit hybrid model + predict
# -----------------------
def fit_full_model_hybrid(seed: int, epochs_fixed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = HybridStringBilinearDeltaModel(
        d_base_p=Zt.shape[1],
        d_base_o=Uo_t.shape[1],
        d_seq=U_seq_t.shape[1],
        d_net=U_net_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT,
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    idx = np.arange(N)
    for epoch in range(1, int(epochs_fixed) + 1):
        model.train()
        np.random.shuffle(idx)

        for start in range(0, N, BATCH_GENES):
            b = idx[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(
                Zt.index_select(0, b_t),
                Z_seq_t.index_select(0, b_t),
                Z_net_t.index_select(0, b_t),
                Uo_t,
                U_seq_t,
                U_net_t,
                Z_seq_mask_t.index_select(0, b_t),
                Z_net_mask_t.index_select(0, b_t),
                U_seq_mask_t,
                U_net_mask_t,
            )

            dt_b = Yt.index_select(0, b_t)
            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % 10 == 0 or epoch == epochs_fixed:
            # quick train score proxy (not OOF)
            model.eval()
            with torch.no_grad():
                tr_pred = model(
                    Zt, Z_seq_t, Z_net_t,
                    Uo_t, U_seq_t, U_net_t,
                    Z_seq_mask_t, Z_net_mask_t,
                    U_seq_mask_t, U_net_mask_t
                ).detach().cpu().numpy().astype(np.float32)
                tr_pred_s = apply_shrink(tr_pred, delta_baseline, alpha_use)
                tr_score = score_delta(Y, tr_pred_s).score
            print(f"[REFIT][TRAIN] seed={seed} epoch={epoch:4d} train_score={tr_score:.6f} loss={float(loss):.6f}")

    return model

# Choose epochs/alpha from the best hybrid run above (edit if you want)
# If you didn't run the compare cell yet, set these manually.
epochs_fixed = None
alpha_use = None

# Try to auto pick from results2
if isinstance(results2, list) and len(results2) > 0:
    best_row = sorted(results2, key=lambda d: d["oof_score"], reverse=True)[0]
    epochs_fixed = int(best_row["median_best_epoch"])
    alpha_use = float(best_row["oof_alpha"])
    print("[auto] picked from best CV row:", best_row["name"], "epochs_fixed=", epochs_fixed, "alpha=", alpha_use)

if epochs_fixed is None:
    epochs_fixed = 200
if alpha_use is None:
    alpha_use = 0.78

# Set training embeddings to linksB base (match submission base)
set_embeddings(U_out_base, Z_train_base_links_B)

MODEL_SEEDS = [6, 67, 6767]
print("[SUBMIT] epochs_fixed=", epochs_fixed, "alpha=", alpha_use, "seeds=", MODEL_SEEDS)

pred_list = []
for s in MODEL_SEEDS:
    m = fit_full_model_hybrid(seed=int(s), epochs_fixed=epochs_fixed)
    m.eval()
    with torch.no_grad():
        p = m(
            Z_base_sub_t,
            Z_seq_sub_t,
            Z_net_sub_t,
            Uo_t,
            U_seq_t,
            U_net_t,
            Z_seq_sub_mask_t,
            Z_net_sub_mask_t,
            U_seq_mask_t,
            U_net_mask_t,
        ).detach().cpu().numpy().astype(np.float32)
    pred_list.append(p)

pred_raw = np.mean(np.stack(pred_list, axis=0), axis=0).astype(np.float32)  # (rows, G)
pred = apply_shrink(pred_raw, delta_baseline, alpha_use).astype(np.float32)

sub_out = df_sub.copy()
sub_out[gene_columns] = pred.astype(np.float64)

out_path = f"submission_hybrid_seqnet_linksB_e{epochs_fixed}_a{alpha_use:.3f}.csv"
# sub_out.to_csv(out_path, index=False, float_format="%.20f")
print("[SUBMIT] ready:", out_path, "shape:", sub_out.shape)
